# CNN Feature Extraction — BreaKHis
**AFFC-Net Base Paper Implementation**

Extracts ResNet-50 spatial feature maps `[N, 196, 1024]` from BreaKHis histopathology images.

**Dataset notes:**
- BreaKHis has a nested folder structure: `BreaKHis_v1/histology_slides/breast/{benign,malignant}/SOB/<subtype>/<slide>/<magnification>/*.png`
- A custom `BreaKHisDataset` is used to correctly walk this structure and assign binary labels (`0 = benign`, `1 = malignant`)
- Labels ARE saved from this notebook as `breakhis_labels_ver2.pt` (integer tensor, 0/1)
- CNN features saved as `breakhis_cnn_features_ver2.pt` with shape `[N, 196, 1024]`

**Same extraction strategy as LC25000 notebook:**
- Direct `Resize(224, 224)` — paper §5.2.1 specifies 224×224 input
- No augmentation at extraction time — features saved deterministically
- ResNet-50 truncated after `layer3` → `[B, 1024, 14, 14]` → reshaped to `[B, 196, 1024]`

In [1]:
import os, time
import torch
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ── GPU selection ────────────────────────────────────────────────────────────
# Change to your GPU index or MIG partition string
DEVICE_ID = "cuda:5"
device = torch.device(DEVICE_ID if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    idx = int(DEVICE_ID.split(":")[-1]) if ":" in DEVICE_ID else 0
    print("GPU:", torch.cuda.get_device_name(idx))

Image.MAX_IMAGE_PIXELS = None  # allow large pathology images


Using device: cuda:5
GPU: NVIDIA H200


## Transform
Paper §5.2.1: *'resizing all input images to 224×224×3'*
No augmentation here — features saved once, deterministically.

In [2]:
# Paper §5.2.1: resize directly to 224×224 (no crop at extraction time)
# Augmentation (random crop / flip) is applied during model training, not here.
transform = transforms.Compose([
    transforms.Resize((224, 224)),          # paper-exact: 224×224×3
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],         # ImageNet stats (ResNet-50 pretrained)
        std=[0.229, 0.224, 0.225]
    )
])
print("Transform defined (224×224, no augmentation — deterministic extraction)")


Transform defined (224×224, no augmentation — deterministic extraction)


## Dataset — BreaKHis Custom Loader

BreaKHis folder structure:
```
BreaKHis_v1/
  histology_slides/
    breast/
      benign/
        SOB/
          adenosis/       ← subtype
            SOB_B_A_14-22549AB/   ← slide
              40X/                ← magnification
                *.png
              100X/
              ...
      malignant/
        SOB/
          ductal_carcinoma/
            ...
```

**Label assignment:** `benign → 0`, `malignant → 1`

All magnification levels (40X, 100X, 200X, 400X) are included.
Total images: ~7,909 across both classes.

In [4]:
import os
for entry in os.walk("BreaKHis_v1"):
    print(entry[0])
    break  # just show top-level subdirs

# then:
print(os.listdir("BreaKHis_v1"))

BreaKHis_v1
['BreaKHis_v1']


In [6]:
from torchvision import datasets

DATA_DIR = "BreaKHis_v1/BreaKHis_v1"

dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

print(f"Total images  : {len(dataset)}")
print(f"Classes found : {dataset.classes}")
print(f"Class→index   : {dataset.class_to_idx}")

Total images  : 7909
Classes found : ['histology_slides']
Class→index   : {'histology_slides': 0}


## ResNet-50 backbone — stop at layer3
Paper §4.1.1: *'after three stages, generated 14×14×1024 feature maps'*

In [7]:
loader = DataLoader(
    dataset,
    batch_size=128,        # large batch — GPU-bound extraction
    shuffle=False,         # keep original order for index alignment with GNN
    num_workers=8,
    pin_memory=True
)

# ResNet-50 truncated after layer3 → output [B, 1024, 14, 14]
_resnet = models.resnet50(weights="DEFAULT")
cnn_backbone = torch.nn.Sequential(
    _resnet.conv1,
    _resnet.bn1,
    _resnet.relu,
    _resnet.maxpool,
    _resnet.layer1,   # stage 1
    _resnet.layer2,   # stage 2
    _resnet.layer3    # stage 3 → [B, 1024, 14, 14]
    # layer4 NOT included — paper stops here
).to(device).eval()

print("CNN backbone ready  (ResNet-50, stops at layer3)")
print(f"Expected output shape per batch: [B, 1024, 14, 14]")


CNN backbone ready  (ResNet-50, stops at layer3)
Expected output shape per batch: [B, 1024, 14, 14]


## Feature extraction loop
Reshape `[B, 1024, 14, 14]` → `[B, 196, 1024]` (paper Eq.4, §5.2.2 shape for AFF input)

In [8]:
cnn_features_list = []
labels_list       = []

t0 = time.time()
with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        feats  = cnn_backbone(images)           # [B, 1024, 14, 14]
        # Reshape to [B, 196, 1024] — matches GNN output shape for AFF fusion
        feats  = feats.flatten(2).permute(0, 2, 1).contiguous()  # [B, 196, 1024]
        cnn_features_list.append(feats.cpu())
        labels_list.append(labels)              # keep labels aligned with features
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx:>4}/{len(loader)}  shape={feats.shape}")

torch.cuda.synchronize()
elapsed = time.time() - t0
print(f"\nExtraction complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")


  Batch    0/62  shape=torch.Size([128, 196, 1024])
  Batch   10/62  shape=torch.Size([128, 196, 1024])
  Batch   20/62  shape=torch.Size([128, 196, 1024])
  Batch   30/62  shape=torch.Size([128, 196, 1024])
  Batch   40/62  shape=torch.Size([128, 196, 1024])
  Batch   50/62  shape=torch.Size([128, 196, 1024])
  Batch   60/62  shape=torch.Size([128, 196, 1024])

Extraction complete in 55.7s (0.9 min)


In [9]:
cnn_features = torch.cat(cnn_features_list, dim=0)
labels       = torch.cat(labels_list, dim=0)

print(f"Final CNN feature tensor shape : {cnn_features.shape}")
print(f"Final labels tensor shape      : {labels.shape}")
print(f"Label distribution — benign(0): {(labels==0).sum().item()}, "
      f"malignant(1): {(labels==1).sum().item()}")

assert cnn_features.shape[0] == labels.shape[0], "Feature/label count mismatch!"
assert cnn_features.shape[1] == 196,  "Expected 196 spatial nodes (14×14)"
assert cnn_features.shape[2] == 1024, "Expected 1024 feature channels"
print("Shape assertions passed.")


Final CNN feature tensor shape : torch.Size([7909, 196, 1024])
Final labels tensor shape      : torch.Size([7909])
Label distribution — benign(0): 7909, malignant(1): 0
Shape assertions passed.


## Save
Both CNN features and labels are saved from this notebook.
Labels are binary: `0 = benign`, `1 = malignant`.

In [12]:
torch.save(cnn_features, "breakhis_cnn_features_ver2.pt")
torch.save(labels,       "breakhis_cnn_labels_ver2.pt")

print(f"Saved → breakhis_cnn_features_ver2.pt  {cnn_features.shape}")
print(f"Saved → breakhis_labels_ver2.pt        {labels.shape}")
print("Done. Labels: 0 = benign, 1 = malignant.")


Saved → breakhis_cnn_features_ver2.pt  torch.Size([7909, 196, 1024])
Saved → breakhis_labels_ver2.pt        torch.Size([7909])
Done. Labels: 0 = benign, 1 = malignant.
